# S3.21 - badRecordsPath
**Date completed:** November 2026  
**Status:** Completed   
**Key concept:** Bad records are written to a separate path, with the reason, so they can be reviewed

In [0]:
# ============================================================
# S3.21 — Cell 2
# badRecordsPath: Capture Malformed CSV Records to Files
#
# Goal:
# 1. Read a CSV with a malformed row using badRecordsPath.
# 2. See which rows the returned DataFrame contains.
# 3. Inspect every readable diagnostic line written under
#    badRecordsPath.
# 4. Test the extra-token row separately so its behaviour is
#    not confused with the invalid integer row.
# 5. Compare predictions with actual workspace behaviour.
# 6. Reconcile returned rows with known captured source rows.
#
# Dataset:
# Controlled CSV files stored in a Unity Catalog Volume.
#
# Main source:
#
# Row 3:
# 3,Pune,abc
# → qty expects IntegerType
# → malformed typed value
#
# Row 4:
# 4,Chennai,40,EXTRA
# → extra-token behaviour is NOT assumed
# → tested independently later in this cell
#
# Important:
# We validate data behaviour, not the number of physical files
# Spark happens to create.
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)


print("=== S3.21: badRecordsPath ===")
print(f"Spark version: {spark.version}")
print()


# ------------------------------------------------------------
# Step 1 — Create Volume and Define Paths
# ------------------------------------------------------------

spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.default.s3_21_files
""")


BASE_PATH = (
    "/Volumes/workspace/default/"
    "s3_21_files"
)

CSV_PATH = (
    f"{BASE_PATH}/orders.csv"
)

BAD_PATH = (
    f"{BASE_PATH}/bad_records"
)

ROW4_PATH = (
    f"{BASE_PATH}/row4_only.csv"
)

ROW4_BAD_PATH = (
    f"{BASE_PATH}/bad_records_row4"
)


print(f"Base path       : {BASE_PATH}")
print(f"CSV path        : {CSV_PATH}")
print(f"Bad-record path : {BAD_PATH}")
print()

In [0]:
# ------------------------------------------------------------
# Step 2 — Clean Previous Test Output
#
# This experiment must be isolated.
#
# If cleanup fails, stop instead of accidentally reading
# bad-record files from an earlier run.
# ------------------------------------------------------------

print("=== STEP 2: CLEAN PREVIOUS OUTPUT ===")


for path in [
    BAD_PATH,
    ROW4_BAD_PATH
]:

    try:

        dbutils.fs.rm(
            path,
            True
        )

        print(
            f"Cleanup completed: {path}"
        )

    except Exception as e:

        print(
            f"Cleanup failed for: {path}"
        )

        print(
            f"{type(e).__name__}: "
            f"{str(e)[:300]}"
        )

        raise


print()

In [0]:
# ------------------------------------------------------------
# Step 3 — Create Controlled CSV
#
# Header:
#
# id,city,qty
#
# Row 1 → valid
# Row 2 → valid
#
# Row 3:
# qty = abc
# → cannot be parsed as IntegerType
#
# Row 4:
# extra CSV token
# → behaviour tested, not assumed
#
# Row 5 → valid
# ------------------------------------------------------------

print("=== STEP 3: CREATE SOURCE CSV ===")


lines = [

    "id,city,qty",

    "1,Delhi,10",

    "2,Mumbai,20",

    "3,Pune,abc",

    "4,Chennai,40,EXTRA",

    "5,Delhi,50",
]


source_rows = (
    len(lines) - 1
)


dbutils.fs.put(
    CSV_PATH,
    "\n".join(lines),
    True
)


print(
    f"Source data rows: {source_rows}"
)

print(
    f"CSV created at : {CSV_PATH}"
)

print()

In [0]:
# ------------------------------------------------------------
# Step 4 — Explicit Schema
# ------------------------------------------------------------

schema = StructType([

    StructField(
        "id",
        IntegerType(),
        True
    ),

    StructField(
        "city",
        StringType(),
        True
    ),

    StructField(
        "qty",
        IntegerType(),
        True
    )
])


print("=== STEP 4: SCHEMA ===")

print(schema)

print()

In [0]:
# ------------------------------------------------------------
# Step 5 — Prediction Before Execution
#
# This is a prediction, NOT an observed result.
#
# We predict:
#
# Row 3:
# → redirected to badRecordsPath
#
# Row 4:
# → predicted to remain
#
# Therefore:
#
# predicted returned IDs:
# 1, 2, 4, 5
#
# predicted returned rows:
# 4
#
# Actual behaviour will be checked below.
# ------------------------------------------------------------

print("=== STEP 5: PREDICTION ===")


predicted_returned_rows = 4

predicted_ids = [
    1,
    2,
    4,
    5
]


print(
    f"Predicted returned rows: "
    f"{predicted_returned_rows}"
)

print(
    f"Predicted returned IDs : "
    f"{predicted_ids}"
)

print()

In [0]:
# ------------------------------------------------------------
# Step 6 — Read with badRecordsPath
#
# Explicit PERMISSIVE mode keeps the experiment clear.
#
# collect() is intentional here because:
#
# - only five source rows exist
# - we want all fields parsed
# - we want to inspect every returned row
#
# The ACTION is inside try/except because parsing happens
# during execution.
# ------------------------------------------------------------

print(
    "=== STEP 6: READ WITH badRecordsPath ==="
)


read_succeeded = False

returned_rows = []


try:

    df_bad_path = (

        spark.read

        .option(
            "header",
            "true"
        )

        .option(
            "mode",
            "PERMISSIVE"
        )

        .option(
            "badRecordsPath",
            BAD_PATH
        )

        .schema(
            schema
        )

        .csv(
            CSV_PATH
        )
    )


    returned_rows = (
        df_bad_path.collect()
    )


    read_succeeded = True


    print(
        f"Rows returned: "
        f"{len(returned_rows)}"
    )


    for row in returned_rows:

        print(row)


except Exception as e:

    print(
        "Read failed."
    )

    print(
        f"Exception type: "
        f"{type(e).__name__}"
    )

    print(
        f"Message preview: "
        f"{str(e)[:400]}"
    )


print()

In [0]:
# ------------------------------------------------------------
# Step 7 — Returned IDs
# ------------------------------------------------------------

returned_ids = []


if read_succeeded:

    returned_ids = sorted(

        row["id"]

        for row in returned_rows

        if row["id"] is not None
    )


    print(
        "=== STEP 7: RETURNED IDS ==="
    )


    print(
        f"Returned IDs: "
        f"{returned_ids}"
    )

    print()

In [0]:
# ------------------------------------------------------------
# Step 8 — Prediction vs Actual
#
# Print differences.
#
# Do NOT abort just because the runtime differs from the
# prediction — the runtime behaviour is what this experiment
# is trying to learn.
# ------------------------------------------------------------

if read_succeeded:

    print(
        "=== STEP 8: PREDICTION vs ACTUAL ==="
    )


    print(
        f"Returned rows : "
        f"{len(returned_rows)} "
        f"(predicted {predicted_returned_rows})"
    )

    print(
        f"Returned IDs  : "
        f"{returned_ids} "
        f"(predicted {predicted_ids})"
    )

    print(
        f"ID 3 returned : "
        f"{3 in returned_ids} "
        f"(predicted False)"
    )

    print(
        f"ID 4 returned : "
        f"{4 in returned_ids} "
        f"(predicted True)"
    )

    print()

In [0]:
# ------------------------------------------------------------
# Step 9 — Recursively List badRecordsPath
#
# Do NOT swallow listing failures.
#
# Otherwise:
#
# files found = 0
#
# could incorrectly hide:
#
# permission failure
# bad path
# unsupported operation
# listing failure
# ------------------------------------------------------------

def list_files_recursive(path):

    found = []


    for item in dbutils.fs.ls(
        path
    ):

        if item.path.endswith("/"):

            found.extend(

                list_files_recursive(
                    item.path
                )
            )

        else:

            found.append(
                item.path
            )


    return found


print(
    "=== STEP 9: FILES IN badRecordsPath ==="
)


bad_files = []

bad_path_listing_succeeded = False


try:

    bad_files = (
        list_files_recursive(
            BAD_PATH
        )
    )


    bad_path_listing_succeeded = True


    print(
        f"Physical files found: "
        f"{len(bad_files)}"
    )


    for file_path in bad_files:

        print(
            file_path
        )


except Exception as e:

    print(
        "Unable to list badRecordsPath."
    )

    print(
        f"Exception type: "
        f"{type(e).__name__}"
    )

    print(
        f"Message preview: "
        f"{str(e)[:400]}"
    )


print()

In [0]:
# ------------------------------------------------------------
# Step 10 — Read Every Diagnostic File as Raw Text
#
# We inspect raw text first because we do not want to assume
# the internal diagnostic-file schema.
# ------------------------------------------------------------

print(
    "=== STEP 10: BAD-RECORD FILE CONTENT ==="
)


bad_file_lines = []


for file_path in bad_files:

    file_name = (

        file_path

        .rstrip("/")

        .split("/")[-1]
    )


    # Skip common metadata/control files.
    if (
        file_name.startswith("_")
        or
        file_name.startswith(".")
    ):

        continue


    try:

        text_rows = (

            spark.read

            .text(
                file_path
            )

            .collect()
        )


        for row in text_rows:

            value = (
                row["value"]
            )


            bad_file_lines.append(
                value
            )


            print(
                value[:500]
            )


    except Exception as e:

        print(
            f"Could not read: "
            f"{file_path}"
        )

        print(
            f"{type(e).__name__}: "
            f"{str(e)[:200]}"
        )


print()

In [0]:
# ------------------------------------------------------------
# Step 11 — Identify Known Source Records in Diagnostic Text
#
# IMPORTANT:
#
# len(bad_file_lines)
#
# means:
#
# number of readable text lines from diagnostic files.
#
# It does NOT automatically mean:
#
# number of malformed source records.
#
# Therefore validate known source records by content.
# ------------------------------------------------------------

print(
    "=== STEP 11: CAPTURED SOURCE RECORDS ==="
)


captured_text = (
    "\n".join(
        bad_file_lines
    )
)


row3_captured = (
    "3,Pune,abc"
    in
    captured_text
)


row4_captured = (
    "4,Chennai,40,EXTRA"
    in
    captured_text
)


captured_known_records = (

    int(
        row3_captured
    )

    +

    int(
        row4_captured
    )
)


print(
    f"Diagnostic text lines read : "
    f"{len(bad_file_lines)}"
)

print(
    f"Row 3 captured             : "
    f"{row3_captured}"
)

print(
    f"Row 4 captured             : "
    f"{row4_captured}"
)

print(
    f"Known source records found : "
    f"{captured_known_records}"
)

print()

In [0]:
# ------------------------------------------------------------
# Step 12 — Isolate Row 4
#
# The main source contains BOTH:
#
# row 3 → bad IntegerType value
# row 4 → extra CSV token
#
# To understand row 4 independently, create another file
# containing ONLY row 4.
# ------------------------------------------------------------

print(
    "=== STEP 12: ROW 4 IN ISOLATION ==="
)


row4_lines = [

    "id,city,qty",

    "4,Chennai,40,EXTRA"
]


dbutils.fs.put(
    ROW4_PATH,
    "\n".join(row4_lines),
    True
)


print(
    f"Row-4-only file: "
    f"{ROW4_PATH}"
)

print()

In [0]:
# ------------------------------------------------------------
# Step 13 — Row 4: Plain PERMISSIVE
#
# No badRecordsPath.
#
# This tells us how the CSV parser itself handles the
# extra-token record in this workspace/runtime.
# ------------------------------------------------------------

print(
    "=== STEP 13: ROW 4 — PLAIN PERMISSIVE ==="
)


row4_plain_succeeded = False

row4_plain_rows = []


try:

    row4_plain_rows = (

        spark.read

        .option(
            "header",
            "true"
        )

        .option(
            "mode",
            "PERMISSIVE"
        )

        .schema(
            schema
        )

        .csv(
            ROW4_PATH
        )

        .collect()
    )


    row4_plain_succeeded = True


    print(
        f"Rows returned: "
        f"{len(row4_plain_rows)}"
    )


    for row in row4_plain_rows:

        print(row)


except Exception as e:

    print(
        f"Exception: "
        f"{type(e).__name__}: "
        f"{str(e)[:300]}"
    )


print()

In [0]:
# ------------------------------------------------------------
# Step 14 — Row 4: PERMISSIVE + badRecordsPath
#
# Same isolated row, now with badRecordsPath.
#
# This lets us compare:
#
# plain parser behaviour
# vs
# badRecordsPath behaviour
# ------------------------------------------------------------

print(
    "=== STEP 14: ROW 4 — WITH badRecordsPath ==="
)


row4_bad_path_succeeded = False

row4_bad_path_rows = []


try:

    row4_bad_path_rows = (

        spark.read

        .option(
            "header",
            "true"
        )

        .option(
            "mode",
            "PERMISSIVE"
        )

        .option(
            "badRecordsPath",
            ROW4_BAD_PATH
        )

        .schema(
            schema
        )

        .csv(
            ROW4_PATH
        )

        .collect()
    )


    row4_bad_path_succeeded = True


    print(
        f"Rows returned: "
        f"{len(row4_bad_path_rows)}"
    )


    for row in row4_bad_path_rows:

        print(row)

except Exception as e:

    print(
        f"Exception: "
        f"{type(e).__name__}: "
        f"{str(e)[:300]}"
    )


print()

In [0]:
# ------------------------------------------------------------
# Step 15 — Inspect Isolated Row-4 badRecordsPath
# ------------------------------------------------------------

print(
    "=== STEP 15: ROW 4 BAD-PATH CONTENT ==="
)


row4_bad_files = []


try:

    row4_bad_files = (
        list_files_recursive(
            ROW4_BAD_PATH
        )
    )


    print(
        f"Physical files found: "
        f"{len(row4_bad_files)}"
    )


    for file_path in row4_bad_files:

        print(
            file_path
        )


except Exception as e:

    print(
        "Unable to list row-4 badRecordsPath."
    )

    print(
        f"{type(e).__name__}: "
        f"{str(e)[:300]}"
    )


row4_bad_lines = []


for file_path in row4_bad_files:

    file_name = (

        file_path

        .rstrip("/")

        .split("/")[-1]
    )


    if (
        file_name.startswith("_")
        or
        file_name.startswith(".")
    ):

        continue


    try:

        for row in (

            spark.read

            .text(
                file_path
            )

            .collect()
        ):

            value = (
                row["value"]
            )


            row4_bad_lines.append(
                value
            )


            print(
                value[:500]
            )


    except Exception as e:

        print(
            f"Could not read {file_path}: "
            f"{type(e).__name__}: "
            f"{str(e)[:200]}"
        )


row4_bad_text = (
    "\n".join(
        row4_bad_lines
    )
)


row4_isolated_captured = (
    "4,Chennai,40,EXTRA"
    in
    row4_bad_text
)


print(
    f"\nIsolated row 4 captured: "
    f"{row4_isolated_captured}"
)

print()

In [0]:
# ------------------------------------------------------------
# Step 16 — Reconciliation
#
# Validate known business/data behaviour.
#
# Do NOT use physical file count as malformed-row count.
# ------------------------------------------------------------

print(
    "=== STEP 16: RECONCILIATION ==="
)


if read_succeeded:

    rows_not_returned = (

        source_rows

        -

        len(
            returned_rows
        )
    )


    print(
        f"Source rows                   : "
        f"{source_rows}"
    )

    print(
        f"Returned rows                 : "
        f"{len(returned_rows)}"
    )

    print(
        f"Rows not returned             : "
        f"{rows_not_returned}"
    )

    print(
        f"Known source records captured : "
        f"{captured_known_records}"
    )

    if (
        rows_not_returned
        ==
        captured_known_records
    ):

        print(
            "Known-row reconciliation matched."
        )

    else:

        print(
            "CHECK: rows not returned do not equal "
            "known captured source records."
        )


print()

In [0]:
# ------------------------------------------------------------
# Step 17 — Production Considerations
#
# badRecordsPath is useful for investigation and learning.
#
# Production design should not automatically treat it as the
# final enterprise quarantine solution.
#
# Databricks documentation describes file-based
# badRecordsPath as non-transactional.
#
# Databricks also documents precedence of badRecordsPath
# over _corrupt_record when bad records are redirected
# externally.
#
# Depending on architecture, modern alternatives can include:
#
# _corrupt_record
# _rescued_data
# Auto Loader
# Lakeflow expectations
# explicit Delta quarantine tables
# ------------------------------------------------------------

print(
    "=== STEP 17: PRODUCTION CONSIDERATIONS ==="
)


print("""
badRecordsPath:

Useful for:
→ parser investigation
→ external diagnostic capture
→ learning malformed-record behaviour


Production cautions:

→ file based
→ non-transactional
→ not automatically the preferred enterprise
  quarantine architecture


Possible modern patterns:

_corrupt_record
_rescued_data
Auto Loader
Lakeflow expectations
Delta quarantine tables


Enterprise pattern:

Capture
   ↓
Inspect
   ↓
Classify
   ↓
Quarantine / remediate
   ↓
Reconcile
   ↓
Monitor
""")

In [0]:
# ------------------------------------------------------------
# Step 18 — Final Observed Results
#
# IMPORTANT:
#
# Do not hard-code claims about what happened.
#
# Print what THIS execution actually observed.
# ------------------------------------------------------------

print(
    "=== STEP 18: FINAL OBSERVED RESULTS ==="
)


print(
    f"Main read succeeded          : "
    f"{read_succeeded}"
)

print(
    f"Source rows                  : "
    f"{source_rows}"
)

print(
    f"Returned rows                : "
    f"{len(returned_rows) if read_succeeded else 'N/A'}"
)

print(
    f"Returned IDs                 : "
    f"{returned_ids if read_succeeded else 'N/A'}"
)

print(
    f"Main bad path list succeeded : "
    f"{bad_path_listing_succeeded}"
)

print(
    f"Physical bad files found     : "
    f"{len(bad_files)}"
)

print(
    f"Diagnostic text lines read   : "
    f"{len(bad_file_lines)}"
)

print(
    f"Row 3 captured               : "
    f"{row3_captured}"
)

print(
    f"Row 4 captured in main test  : "
    f"{row4_captured}"
)

print(
    f"Row 4 plain read succeeded   : "
    f"{row4_plain_succeeded}"
)

print(
    f"Row 4 plain returned rows    : "
    f"{len(row4_plain_rows) if row4_plain_succeeded else 'N/A'}"
)

print(
    f"Row 4 bad-path read succeeded: "
    f"{row4_bad_path_succeeded}"
)

print(
    f"Row 4 bad-path returned rows : "
    f"{len(row4_bad_path_rows) if row4_bad_path_succeeded else 'N/A'}"
)

print(
    f"Row 4 captured in isolation  : "
    f"{row4_isolated_captured}"
)

print()

In [0]:
# ------------------------------------------------------------
# Step 19 — Final Learning Summary
# ------------------------------------------------------------

print(
    "=== FINAL LEARNING SUMMARY ==="
)


print("""
badRecordsPath:

→ captures diagnostic information about malformed
  input records outside the returned DataFrame.


Main lesson:

Do not assume parser behaviour.

Predict
   ↓
Run
   ↓
Inspect returned rows
   ↓
Inspect diagnostic capture
   ↓
Isolate ambiguous cases
   ↓
Reconcile


Important distinction:

Physical diagnostic file count
≠
malformed source-record count


For ambiguous parser behaviour:

Create a controlled one-record test.

That is why row 4 was tested separately.


Production principle:

badRecordsPath is a diagnostic/capture mechanism.

It is not by itself a complete data-quality architecture.


Final pipeline pattern:

Capture
   ↓
Inspect
   ↓
Classify
   ↓
Quarantine / remediate
   ↓
Reconcile
   ↓
Monitor
""")


print(
    "=== S3.21 CELL 2 COMPLETED ==="
)

# S3.21 — Key Takeaways: badRecordsPath

## 1. What badRecordsPath Does

`badRecordsPath` stores information about malformed input records in a separate path so they can be investigated later.

Conceptually:

```text
Source CSV
    ↓
Parser
    ↓
┌─────────────────┬──────────────────────┐
│                 │                      │
Valid records     Malformed records
│                 │
Returned          badRecordsPath
DataFrame         files
```

When `badRecordsPath` captures a malformed record, that record does not appear in the returned DataFrame.

---

## 2. Comparison

| Option | Malformed Row | Trace / Evidence |
|---|---|---|
| `DROPMALFORMED` | Dropped | No retained bad-row payload |
| `_corrupt_record` | Kept in DataFrame | Raw malformed line stored in a column |
| `badRecordsPath` | Redirected outside DataFrame | Diagnostic record stored under the bad-record path |

### Important

When `badRecordsPath` is used, it takes precedence over `_corrupt_record`.

Therefore:

```text
_corrupt_record
→ malformed row remains in DataFrame

badRecordsPath
→ malformed row is written externally
→ malformed row is not returned
```

---

## 3. Expected S3.21 Behaviour

Controlled source:

```text
1,Delhi,10
2,Mumbai,20
3,Pune,abc
4,Chennai,40,EXTRA
5,Delhi,50
```

Predicted malformed record for this experiment:

```text
3,Pune,abc
```

because:

```text
qty expects IntegerType
abc cannot be parsed as INT
```

Predicted reconciliation:

```text
Source rows
= 5

Returned DataFrame rows
= 4

Malformed records redirected
= 1
```

Row:

```text
4,Chennai,40,EXTRA
```

is predicted to remain under the CSV behaviour being tested, with the extra token outside the supplied schema discarded.

This prediction must be verified against the actual notebook output.

---

## 4. Lazy Evaluation

Creating the DataFrame:

```python
df = (
    spark.read
    .option("badRecordsPath", BAD_PATH)
    .schema(schema)
    .csv(CSV_PATH)
)
```

does not necessarily parse all records immediately.

The parsing work occurs when Spark executes the plan.

Example:

```python
try:
    rows = df.collect()

except Exception as e:
    ...
```

### Important

The **action that triggers parsing** must be inside the `try`.

Do not say that `spark.read.csv(...)` itself must always be inside the `try`.

---

## 5. What to Validate

Do not validate only:

```text
number of physical files created
```

Spark controls physical output layout, so one malformed record does not necessarily mean exactly one physical file.

Validate the actual data behaviour:

```text
1. Source row count
2. Returned DataFrame row count
3. Malformed row missing from returned DataFrame
4. Expected valid rows still present
5. Bad-record content captured externally
6. Failure reason / diagnostic information
```

---

## 6. Do Not Hide Path Errors

Avoid patterns such as:

```python
try:
    ...
except Exception:
    pass
```

because:

```text
Files found = 0
```

could mean:

```text
No bad records were written
```

or:

```text
The path could not be listed
Permission failed
The operation failed
```

Those are different outcomes.

Always expose the listing or access error.

---

## 7. Reconciliation

For this exercise:

```text
Source rows
      =
Returned rows
      +
Malformed rows redirected
```

Predicted:

```text
5 = 4 + 1
```

This reconciliation is more important than the number of physical files created under `badRecordsPath`.

Do not assume:

```text
number of diagnostic text lines
=
number of malformed source records
```

Validate known source records by their actual captured content.

---

## 8. Production Caveats

`badRecordsPath` is a file-based mechanism.

Databricks documents important limitations:

- It is non-transactional.
- It can produce inconsistent results in some failure scenarios.
- It should not automatically be treated as the preferred enterprise quarantine architecture.

For Auto Loader, Databricks recommends using corrupt-record handling instead of relying on `badRecordsPath` because file-based capture can introduce race-condition concerns.

Depending on the architecture, other patterns may include:

```text
_corrupt_record
_rescued_data
Auto Loader
Lakeflow expectations
explicit Delta quarantine tables
```

---

## 9. S3.20 vs S3.21

### S3.20

```text
PERMISSIVE + _corrupt_record
```

Malformed row:

```text
remains in returned DataFrame
+
raw source line available in a column
```

### S3.21

```text
PERMISSIVE + badRecordsPath
```

Malformed row:

```text
redirected to external files
+
not present in returned DataFrame
```

---

## 10. Testing Ambiguous Parser Behaviour

If one source file contains multiple different malformed conditions, isolate them.

Example:

```text
Main file:
3,Pune,abc
4,Chennai,40,EXTRA
```

To understand the extra-token behaviour independently, create a second controlled source containing only:

```text
4,Chennai,40,EXTRA
```

Then test:

```text
Plain PERMISSIVE
vs
PERMISSIVE + badRecordsPath
```

This prevents one malformed condition from hiding or confusing another.

### Learning Pattern

```text
Predict
   ↓
Run
   ↓
Inspect
   ↓
Isolate ambiguous case
   ↓
Re-run
   ↓
Conclude from evidence
```

---

## 11. Production Flow

```text
Capture
   ↓
Inspect
   ↓
Classify
   ↓
Quarantine / remediate
   ↓
Reconcile
   ↓
Monitor
```

`badRecordsPath` is one possible capture mechanism.

It is not the entire data-quality solution.

---

## 12. Observed in Our Workspace

Not verified yet:

```text
Spark version                     = ______

Main read succeeded               = ______

Source rows                       = 5

Returned DataFrame rows           = ______

Returned IDs                      = ______

ID 3 returned                     = ______

ID 4 returned                     = ______

badRecordsPath accessible         = ______

Physical files found              = ______

Diagnostic text lines read        = ______

Captured "3,Pune,abc"             = ______

Captured "4,Chennai,40,EXTRA"     = ______

Captured failure reason           = ______
```

### Isolated Row 4 Test

```text
Plain PERMISSIVE succeeded        = ______

Plain PERMISSIVE returned rows    = ______

badRecordsPath read succeeded     = ______

badRecordsPath returned rows      = ______

Row 4 captured externally         = ______
```

Do not mark S3.21 complete until these values come from the actual notebook output.

---

## 13. Related Interview Questions and Answers

### Q1. What is the difference between `_corrupt_record` and `badRecordsPath`?

`_corrupt_record` keeps malformed-record information inside the returned DataFrame.

`badRecordsPath` redirects malformed-record information to external files and excludes those malformed rows from the returned DataFrame.

---

### Q2. What is the difference between `DROPMALFORMED` and `badRecordsPath`?

`DROPMALFORMED` removes malformed rows from the DataFrame without retaining the malformed source payload in the DataFrame.

`badRecordsPath` redirects diagnostic information about malformed records to an external path for investigation.

---

### Q3. Why should we not validate the number of bad-record files?

Because Spark controls physical file layout.

The important validation is whether the expected malformed data was captured, not whether Spark created exactly one physical file.

---

### Q4. Why does the action belong inside `try/except`?

Because Spark is lazy.

The parser may not process the malformed record until an action such as `collect()` executes the plan.

---

### Q5. Is `badRecordsPath` a complete DQ solution?

No.

It captures malformed input externally, but the pipeline still needs:

```text
inspection
classification
quarantine or remediation
reconciliation
monitoring
```

---

### Q6. Is `badRecordsPath` transactional?

No.

The file-based mechanism is non-transactional, so it should not automatically be treated as the preferred enterprise quarantine design.

---

### Q7. Why isolate row 4 into a separate file?

Because the main source contains two different parsing conditions:

```text
abc → IntegerType conversion problem
extra token → token-count/schema problem
```

Testing row 4 separately allows us to observe its behaviour without interference from row 3.

---

### Q8. Why should we not use `len(bad_file_lines)` as the number of bad source records?

Because diagnostic files may contain one or more text lines per diagnostic entry.

Therefore:

```text
diagnostic text-line count
≠
guaranteed malformed-record count
```

Validate known source records by their captured content instead.

---

### Q9. Which is preferred for modern production ingestion?

It depends on the architecture.

Possible approaches include:

```text
_corrupt_record
_rescued_data
Auto Loader
Lakeflow expectations
Delta quarantine tables
```

The correct choice depends on ingestion requirements, recovery strategy, observability, and governance.

---

## 14. Core Understanding

```text
Need malformed record inside DataFrame?
→ _corrupt_record

Need malformed record redirected externally?
→ badRecordsPath

Need malformed row removed?
→ DROPMALFORMED

Need malformed input to fail ingestion?
→ FAILFAST

Need to understand ambiguous parser behaviour?
→ isolate the record and test it separately

Need enterprise DQ?
→ capture
→ inspect
→ classify
→ quarantine / remediate
→ reconcile
→ monitor
```

**S3.21 — badRecordsPath remains open until actual workspace output is verified.**